# FLIC: Data Analysis Part 2 — Grouped Analysis and Treatments
**Scott Pletcher and the Pletcher Lab**

---

## Overview

This notebook covers treatment-level analysis: computing feeding summaries across multiple DFMs, grouping by treatment, running statistical tests, and visualising timecourses.  It is the Python port of the R document *GroupedAnalysis.Rmd*.

The key difference from the R workflow is that **the experimental design is embedded in `flic_config.yaml`** rather than a separate CSV file.  When you call `Experiment.load()`, the design is built automatically from the YAML: every chamber on every DFM already knows its treatment.  There is no need to maintain a separate `expDesign.csv` or to create per-DFM parameter lists.

A second difference is the `design_factors` block in the config.  In the test data this defines two independent factors — `paired` (Paired / Unpaired) and `genotype` (Chrim / WCS) — that are crossed to produce the four treatment labels.  These individual factor columns appear automatically in the feeding summary and make two-way ANOVA straightforward.

All examples use the data in `python_test_data/` (8 DFMs, two-well / choice experiment).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

def _find_repo_root(marker: str = "pyproject.toml") -> Path:
    for p in [Path().resolve(), *Path().resolve().parents]:
        if (p / marker).exists():
            return p
    raise RuntimeError(f"Could not find repo root ({marker} not found in any parent).")

repo_root = _find_repo_root()
if str(repo_root.parent) not in sys.path:
    sys.path.insert(0, str(repo_root.parent))

from pyflic import Experiment, DFM, Parameters

project_dir = repo_root / "python_test_data"
exp = Experiment.load(project_dir)
print("DFMs loaded:", sorted(exp.dfms.keys()))

## Experiment Design

In pyflic the design lives entirely inside `flic_config.yaml`.  Under the `dfms:` key each DFM lists its chambers, and each chamber is assigned a treatment name.  There is no separate `ExpDesign.csv`.

The optional `design_factors` block in the YAML defines individual factors whose levels are embedded in the treatment name.  For the test data:

```yaml
design_factors:
  paired:   [Paired, Unpaired]
  genotype: [Chrim, WCS]
```

When `feeding_summary()` is called, `paired` and `genotype` columns are appended automatically — no manual string splitting required.  This makes two-way ANOVA trivial (see §&nbsp;**Statistical Analysis**).

Another key difference from R: `pi_direction` (`"left"` or `"right"`) is set **per DFM** in the YAML rather than via a list of parameter objects.  pyflic applies the correct direction automatically when loading each DFM, so you cannot accidentally misalign a parameters list.

`exp.design.design_table()` returns the full design as a tidy DataFrame.

In [ ]:
exp.design.design_table().sort_values(["Treatment", "DFM", "Chamber"])

In [ ]:
# Factor columns are appended when design_factors is configured in the YAML.
fs_preview = exp.feeding_summary()
print("Treatments:", sorted(fs_preview["Treatment"].unique()))
print("Columns   :", list(fs_preview.columns[:8]))

## Feeding Summary

`exp.feeding_summary()` replaces both forms of R's `Feeding.Summary.Monitors()` — with and without an `expDesign` argument.  Because the design is already embedded in the config, a single call produces a table with one row per (DFM, Chamber) that includes the treatment name and any factor columns.

For two-well (choice) experiments the metric columns come in A/B pairs (`LicksA`, `LicksB`, `EventsA`, `EventsB`, …).  `WellA` and `WellB` are always defined relative to `pi_direction`, so a positive PI always indicates preference for the food placed on the designated side regardless of physical well position.

| R function | Python equivalent |
|---|---|
| `Feeding.Summary.Monitors(monitors, p)` | `exp.feeding_summary()` |
| `Feeding.Summary.Monitors(monitors, p, expDesign=expDesign)` | `exp.feeding_summary()` (design built in) |

**Key arguments:**

| Argument | Default | Description |
|---|---|---|
| `range_minutes` | `(0, 0)` | Analysis window in minutes; `(0, 0)` = entire experiment |
| `transform_licks` | `True` | Apply 0.25-power transform to lick counts to reduce right skew; set `False` for raw counts |

`write_feeding_summary()` saves the result to `project_dir/analysis/feeding_summary.csv`.

In [ ]:
fs = exp.feeding_summary()
fs.head(12)

In [ ]:
# Save to CSV (writes to project_dir/analysis/feeding_summary.csv).
exp.write_feeding_summary()

In [ ]:
# Restrict to the first two hours; use raw (untransformed) lick counts.
fs_2h = exp.feeding_summary(range_minutes=(0, 120), transform_licks=False)
fs_2h.head(6)

## Binned Feeding Summary

`exp.binned_feeding_summary()` groups feeding metrics into fixed-width (or custom) time windows.  It extends the experiment-level feeding summary the same way `BinnedFeeding.Summary.Monitors()` extends `Feeding.Summary.Monitors()` in R.

Two extra columns appear in the result:

| Column | Description |
|---|---|
| `Interval` | Bin boundary string, e.g. `"(0, 30]"` |
| `Minutes` | Bin midpoint in minutes |

| R function | Python equivalent |
|---|---|
| `BinnedFeeding.Summary.Monitors(monitors, p, binsize.min=30)` | `exp.binned_feeding_summary(binsize_min=30, save=False)` |

**Key arguments:**

| Argument | Default | Description |
|---|---|---|
| `binsize_min` | — | Fixed bin width in minutes (mutually exclusive with `bins`) |
| `bins` | — | Explicit bin edges, e.g. `(0, 30, 90, 180, 360)` for unequal bins |
| `save` | `True` | Write result to `project_dir/analysis/binned_feeding_summary.csv`; use `False` for in-memory only |
| `range_minutes` | `(0, 0)` | Analysis window |
| `transform_licks` | `True` | 0.25-power lick transform |

In [ ]:
bfs = exp.binned_feeding_summary(binsize_min=30, save=False)
bfs.head(12)

In [ ]:
# Unequal bins — useful when you want finer resolution early in the experiment.
bfs_custom = exp.binned_feeding_summary(bins=(0, 30, 90, 180, 360), save=False)
bfs_custom[["Treatment", "DFM", "Chamber", "Interval", "Minutes", "LicksA", "LicksB"]].head(8)

## Statistical Analysis

pyflic does not include its own statistical routines; instead it integrates with **`statsmodels`**, which is installed as a dependency.  The formula API (`statsmodels.formula.api`) mirrors R's `aov()` / `lm()` syntax closely.

**Important:** always pass `transform_licks=False` when using lick counts in a statistical model.  The 0.25-power transformation is appropriate for plotting (it reduces right skew), but the raw counts are more appropriate for ANOVA because the residuals are easier to interpret and the degrees of freedom remain correct.

| R expression | Python equivalent |
|---|---|
| `summary(aov(Licks ~ Treatment, data=f.summary$Results))` | `anova_lm(smf.ols("LicksA ~ C(Treatment)", data=fs).fit())` |
| `summary(aov(Licks ~ factor(DFM), data=f.summary$Results))` | `anova_lm(smf.ols("LicksA ~ C(DFM)", data=fs).fit())` |

### One-Way ANOVA Across Treatments

In [ ]:
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

fs = exp.feeding_summary(transform_licks=False)

# One-way ANOVA: effect of Treatment on LicksA (Sucrose licks).
model = smf.ols("LicksA ~ C(Treatment)", data=fs).fit()
print(anova_lm(model))

### DFM as a Factor

A common error — inherited from R — is to include DFM as a predictor without marking it as categorical.  Because DFM IDs are integers, a model that treats them as a continuous variable uses **one degree of freedom** regardless of how many DFMs you have.  The resulting F-statistic tests a linear trend across DFM IDs, which is almost never the intended test.

The correct approach is `C(DFM)` in statsmodels (equivalent to `factor(DFM)` in R), which gives one degree of freedom per DFM and correctly partitions the variance.

In [ ]:
# IMPORTANT: DFM must be treated as categorical, not a continuous integer.
# As an integer (incorrect — 1 degree of freedom):
model_wrong = smf.ols("LicksA ~ DFM", data=fs).fit()
print("DFM as continuous (incorrect — 1 df):")
print(anova_lm(model_wrong))

In [ ]:
# Correct: DFM as a categorical factor — one df per DFM.
model_dfm = smf.ols("LicksA ~ C(DFM)", data=fs).fit()
print("DFM as categorical (correct — one df per DFM):")
print(anova_lm(model_dfm))

### Two-Way ANOVA Using Design Factors

Because the `design_factors` block in `flic_config.yaml` adds individual factor columns (`paired`, `genotype`) to the feeding summary, a two-way ANOVA with interaction requires no string manipulation — just reference the column names in the formula.

In [ ]:
# Two-way ANOVA with interaction using individual design factor columns.
# typ=2 uses Type II sums of squares (recommended when sample sizes are unequal).
model_2way = smf.ols("LicksA ~ C(paired) * C(genotype)", data=fs).fit()
print(anova_lm(model_2way, typ=2))

### Binned ANOVA Across Time Intervals

To test whether the treatment effect varies across time, apply the one-way ANOVA separately to each 30-minute bin.  This mirrors the loop in the R document:

```r
for(i in levels(f.binsummary$Results$Interval)[1:5]){
  print(summary(aov(Licks ~ Treatment, data=subset(...))))
}
```

Note: this approach does not correct for multiple comparisons.  Treat the p-values as exploratory.

In [ ]:
bfs_raw = exp.binned_feeding_summary(binsize_min=30, transform_licks=False, save=False)

for interval in sorted(bfs_raw["Interval"].unique())[:5]:
    subset = bfs_raw[bfs_raw["Interval"] == interval]
    model = smf.ols("LicksA ~ C(Treatment)", data=subset).fit()
    result = anova_lm(model)
    p = result["PR(>F)"].iloc[0]
    print(f"Interval {interval}: F p-value = {p:.4f}")

## Plotting: All Metrics

`exp.plot_feeding_summary()` produces a grid of box-and-jitter plots — one panel per feeding metric — grouped by treatment.  Individual chamber values are shown as jittered dots overlaid on box plots.  It returns a plotnine object.

This replaces R's `DataPlot(f.summary, Type="Licks")` and related calls.  Rather than exposing a `Type` argument to select one metric at a time, pyflic renders every metric in a single figure and lets you restrict the time window instead.

| R function | Python equivalent |
|---|---|
| `DataPlot(f.summary, Type="Licks")` | `exp.plot_feeding_summary()` |

In [ ]:
fig = exp.plot_feeding_summary()
fig.show()

In [ ]:
# Restrict the analysis window to the first two hours.
fig = exp.plot_feeding_summary(range_minutes=(0, 120))
fig.show()

## Plotting: Binned Treatment Timecourses

Two functions visualise how feeding evolves across time bins:

| Function | Description | R equivalent |
|---|---|---|
| `exp.plot_binned_metric_by_treatment(metric, binsize_min)` | Mean ± SEM for **one** metric, one line per treatment | `BinnedDataPlot(f.binsummary, Type="Licks")` |
| `exp.plot_binned_metrics_by_treatment(metrics=[...])` | Grid of subplots, one per metric | `BinnedDataPlot(f.binsummary2, Type="Licks")` |

Both return a matplotlib `Figure`.

### `two_well_mode` — combining wells in choice experiments

For two-well experiments the `two_well_mode` argument controls how WellA and WellB are combined before computing the per-treatment mean:

| Value | Behaviour |
|---|---|
| `"total"` | Sum WellA + WellB licks |
| `"A"` | Show WellA only (e.g. Sucrose) |
| `"B"` | Show WellB only (e.g. Yeast) |
| `"mean_ab"` | Average of WellA and WellB |

### Individual chamber traces

Pass `show_individual_chambers=True` to overlay faint per-chamber lines behind the treatment mean, making between-chamber variability visible.

In [ ]:
# Licks across time — WellA + WellB combined, all treatments.
fig = exp.plot_binned_metric_by_treatment(metric="Licks", binsize_min=30)
fig.show()

In [ ]:
# WellA only (Sucrose) — equivalent to the WellA panel in R's BinnedDataPlot.
fig_a = exp.plot_binned_metric_by_treatment(
    metric="Licks", two_well_mode="A", binsize_min=30
)
fig_a.show()

In [ ]:
# WellB only (Yeast).
fig_b = exp.plot_binned_metric_by_treatment(
    metric="Licks", two_well_mode="B", binsize_min=30
)
fig_b.show()

In [ ]:
# Multiple metrics in a single figure — a pyflic extension not present in R.
fig = exp.plot_binned_metrics_by_treatment(
    metrics=["Licks", "Events", "MedDuration", "PI"],
    binsize_min=30,
)
fig.show()

In [ ]:
# Show individual chamber traces to visualise within-treatment variability.
# This is a pyflic-only feature with no direct R equivalent.
fig = exp.plot_binned_metric_by_treatment(
    metric="LicksA",
    binsize_min=30,
    show_individual_chambers=True,
)
fig.show()

## DFM-Level Inspection

Before interpreting treatment effects it is worth checking for DFM-level outliers — situations where one monitor records systematically higher or lower feeding than the others.  These effects can mask or inflate treatment differences.

The cumulative lick plot for an individual DFM (`dfm.plot_cumulative_licks()`) is the most direct diagnostic.  Looping over all DFMs and comparing the slopes and final totals across monitors is equivalent to R's `PlotTotalLicks.Monitors(monitors, p)`.

| R function | Python equivalent |
|---|---|
| `PlotTotalLicks.Monitors(monitors, p)` | Loop over `exp.get_dfm(id).plot_cumulative_licks()` |

In [ ]:
# Cumulative lick plot for each DFM — useful for spotting DFM-level outliers.
for dfm_id in sorted(exp.dfms.keys()):
    fig = exp.get_dfm(dfm_id).plot_cumulative_licks()
    fig.show()

---

## Function Reference

Listed below are the principal functions introduced in this notebook with their signatures.

### `Experiment` — treatment-level analysis

```python
exp.design.design_table()                   # → pd.DataFrame

exp.feeding_summary(
    *, range_minutes=(0, 0),
    transform_licks=True,
)                                            # → pd.DataFrame

exp.write_feeding_summary(path=None, ...)    # → Path

exp.binned_feeding_summary(
    *,
    bins=None,                               # explicit bin edges (mutually exclusive with binsize_min)
    binsize_min=None,                        # fixed bin width in minutes
    range_minutes=(0, 0),
    transform_licks=True,
    path=None,
    save=True,
)                                            # → pd.DataFrame

exp.plot_feeding_summary(
    *, range_minutes=(0, 0),
    transform_licks=True,
    ncols=None, figsize=None,
)                                            # → plotnine.ggplot

exp.plot_binned_metric_by_treatment(
    *, metric="Licks",
    two_well_mode="total",                   # "total" | "A" | "B" | "mean_ab"
    binsize_min=30.0,
    range_minutes=(0, 0),
    transform_licks=True,
    show_sem=True,
    show_individual_chambers=False,
    figsize=(10, 4),
)                                            # → matplotlib Figure

exp.plot_binned_metrics_by_treatment(
    *, metrics=("Licks", "Events", "MedDuration"),
    two_well_mode="total",
    binsize_min=30.0,
    range_minutes=(0, 0),
    transform_licks=True,
    show_sem=True,
    show_individual_chambers=False,
    ncols=2,
    figsize=None,
)                                            # → matplotlib Figure
```